# Financial MCQ Benchmark: Zero-Shot Evaluation

Multi-dataset, multilingual **zero-shot** benchmarking on financial MCQ datasets.

**Datasets:**
- CFA / CPA (English)
- ES-MultiFinQA (Spanish)
- Plutus-MultiFinQA (multilingual)
- BhashaBench Finance (English + Hindi)
- Arabic Accounting MCQ (Arabic)
- Arabic Business MCQ (Arabic)

**Models evaluated:**
1. `microsoft/Phi-3.5-mini-instruct` (~3.8B)
2. `meta-llama/Meta-Llama-3-8B-Instruct` (~8B)
3. `meta-llama/Llama-2-13b-chat-hf` (~13B)

**Metric**: Accuracy — proportion of correctly identified answer labels.  
**Answer format**: index `0–3` (0 = A, 1 = B, 2 = C, 3 = D).

## Cell 1 — Install Required Libraries

In [ ]:
!pip install transformers datasets peft bitsandbytes accelerate
!pip install sentencepiece protobuf evaluate scikit-learn tqdm
!pip install sacremoses langdetect

## Cell 2 — Load All Datasets

Each dataset is standardized to:
- `question`: `str`
- `choices`: list of 4 strings
- `answer`: **`int` (0–3)** — position of the correct option in `choices` (0=A, 1=B, 2=C, 3=D)
- `source`: `str` — dataset identifier

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Load token securely
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")

# Login
login(token=hf_token)

In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets
import pandas as pd
import random

# ── CFA / CPA ──────────────────────────────────────────────────────────────
dataset = load_dataset("Tomas08119993/finmmeval-cfa-cpa", streaming=True)
train = dataset["train"]

cfa_cpa_data = []
it = iter(train)
for i in range(700):          # fetch extra to ensure ≥600 valid rows
    try:
        row = next(it)
        gold = row["gold"]
        if isinstance(gold, list):
            gold = random.choice(gold)   # pick randomly from multi-label answers
        gold = int(gold)
        choices = list(row["choices"])
        if gold < 0 or gold >= len(choices):
            continue
        cfa_cpa_data.append({
            "question": row["text"],
            "choices":  choices,
            "answer":   gold,        # index 0–3
            "source":   "cfa_cpa",
        })
        if len(cfa_cpa_data) >= 600:
            break
    except Exception as e:
        print(f"Skipping row {i}: {e}")
        continue

print(f"CFA/CPA loaded: {len(cfa_cpa_data)} rows")
if cfa_cpa_data:
    print("Sample:", cfa_cpa_data[0])

Loaded rows: 298
{'question': '下列关于管理层编制财务报告的说法中，正确的有（）。', 'choices': ['a', 'b', 'c', 'd'], 'answer': ['b', 'c', 'd']}


In [ ]:
# ── ES-MultiFinQA ──────────────────────────────────────────────────────────
dataset = load_dataset("TheFinAI/flare-es-multifin")
split = dataset["test"]

def process_es(row):
    gold = row["gold"]
    if isinstance(gold, list):
        gold = random.choice(gold)
    return {
        "question": row["text"],
        "choices":  list(row["choices"]),
        "answer":   int(gold),       # index 0–3
        "source":   "es_multifin",
    }

es_multifin_dataset = split.map(process_es, remove_columns=split.column_names)
print(f"ES-MultiFinQA loaded: {len(es_multifin_dataset)} rows")
print("Sample:", es_multifin_dataset[0])

--- Processed Row 0 ---
{'answer': ['Industria'], 'choices': ['Negocios y Gestión', 'Finanzas', 'Gobierno y Control', 'Industria', 'Impuestos y Contabilidad', 'Tecnología'], 'question': 'Gracias - Transporte y logística'}


In [ ]:
# ── Plutus-MultiFinQA ──────────────────────────────────────────────────────
dataset = load_dataset("TheFinAI/plutus-multifin")

def process_plutus(row):
    gold = row["gold"]
    if isinstance(gold, list):
        gold = random.choice(gold)
    return {
        "question": row["text"],
        "choices":  list(row["choices"]),
        "answer":   int(gold),       # index 0–3
        "source":   "plutus",
    }

plutus_splits = [
    dataset[split].map(process_plutus, remove_columns=dataset[split].column_names)
    for split in ["train", "test", "validation"]
]
plutus_dataset = concatenate_datasets(plutus_splits)
print(f"Plutus-MultiFinQA loaded: {len(plutus_dataset)} rows (all splits merged)")
print("Sample:", plutus_dataset[0])

Total rows after merging: 268
--- Processed Row 0 ---
{'answer': 'Επιχειρήσεις & Διοίκηση', 'choices': ['Φορολογία & Λογιστική', 'Επιχειρήσεις & Διοίκηση', 'Οικονομικά', 'Βιομηχανία', 'Τεχνολογία', 'Κυβέρνηση & Έλεγχοι'], 'question': 'Αναστολή συμβάσεων εργασίας Αυγούστου'}


In [ ]:
# ── BhashaBench Finance (EN + HI) ──────────────────────────────────────────
en_data = load_dataset("bharatgenai/BhashaBench-Finance", "English", split="test")
hi_data = load_dataset("bharatgenai/BhashaBench-Finance", "Hindi",   split="test")

_LETTER_TO_IDX = {"A": 0, "B": 1, "C": 2, "D": 3}

def process_bhasha(row):
    choices = [row["option_a"], row["option_b"], row["option_c"], row["option_d"]]
    ans_idx = _LETTER_TO_IDX.get(str(row["correct_answer"]).strip().upper(), 0)
    return {
        "question": row["question"],
        "choices":  choices,
        "answer":   ans_idx,         # index 0–3
        "source":   "bhasha_bench",
    }

bhasha_dataset = concatenate_datasets([
    en_data.map(process_bhasha, remove_columns=en_data.column_names),
    hi_data.map(process_bhasha, remove_columns=hi_data.column_names),
])
print(f"BhashaBench Finance loaded: {len(bhasha_dataset)} rows (EN+HI)")
print("Sample:", bhasha_dataset[0])

English/test-00000-of-00001.parquet:   0%|          | 0.00/4.09M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/13451 [00:00<?, ? examples/s]

Hindi/test-00000-of-00001.parquet:   0%|          | 0.00/1.75M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/5982 [00:00<?, ? examples/s]

Map:   0%|          | 0/13451 [00:00<?, ? examples/s]

Map:   0%|          | 0/5982 [00:00<?, ? examples/s]

Total Combined Rows: 19433
--- Sample Row ---
{'question': 'In the following number series. One number is wrong. Find the wrong number of the series? 3, 4, 12, 38, 103, 228', 'choices': ['103', '12', '38', '228'], 'answer': '38'}


In [ ]:
# ── Arabic Accounting MCQ (train + eval merged) ────────────────────────────
def _process_arabic(row, source_name):
    """Shared processor for all Arabic SahmBenchmark datasets."""
    gold = row["gold"]
    if isinstance(gold, list):
        gold = random.choice(gold)
    return {
        "question": row["text"],
        "choices":  list(row["choices"]),
        "answer":   int(gold),       # index 0–3
        "source":   source_name,
    }

acc_train = load_dataset("SahmBenchmark/arabic-accounting-mcq_train")["train"]
acc_eval  = load_dataset("SahmBenchmark/arabic-accounting-mcq_eval")["test"]

arabic_accounting_dataset = concatenate_datasets([
    acc_train.map(lambda r: _process_arabic(r, "arabic_accounting"), remove_columns=acc_train.column_names),
    acc_eval.map( lambda r: _process_arabic(r, "arabic_accounting"), remove_columns=acc_eval.column_names),
])
print(f"Arabic Accounting loaded: {len(arabic_accounting_dataset)} rows (train+eval)")
print("Sample:", arabic_accounting_dataset[0])

{'answer': ['c'],
 'choices': ['a', 'b', 'c', 'd', 'e'],
 'question': 'السؤال: حالة عملية رقم (2)\n\nفيما يلي البيانات الخاصة بشركة (الهلال) المصرية، والمتعلقة بمعاملتين قامت بإبرامهما خلال عام 2018م مع شركتين أحدهما أمريكية والأخرى سعودية:\n\n**المعاملة الأولى:**\nفي 2018/11/1م أستوردت الشركة بضاعة من شركة (Nabisco) الأمريكية بمبلغ 100,000 دولار أمريكي، وذلك على الحساب بحيث تدفع القيمة للمصدر الأمريكي في 2019/4/1م. ولأغراض التحوط ضد تقلب سعر صرف الدولار الأمريكي مقابل الجنيه المصري خلال فترة المعاملة، فقد قامت الشركة المصرية بإبرام عقد صرف آجل في 2018/11/1م (لمدة 5 شهور) مع بنك القاهرة - فرع الزقازيق، لشراء 100,000 دولار أمريكي يوفرها البنك في 2019/4/1م بسعر صرف يبلغ 15.75 جنيه مصري لكل دولار أمريكي.\n\n**المعاملة الثانية:**\nفي 2018/11/1م صدرت الشركة بضاعة إلى شركة (ينبع) السعودية بمبلغ 100,000 ريال سعودي، وذلك على الحساب بحيث تحصل القيمة من المستورد السعودي في 2019/4/1م. ولأغراض التحوط ضد تقلب سعر صرف الريال السعودي مقابل الجنيه المصري خلال فترة المعاملة، فقد قامت الشركة المصرية بإب

In [ ]:
# ── Arabic Business MCQ (train + eval merged) ──────────────────────────────
# NOTE: _process_arabic is defined in the Arabic Accounting cell above — run that first.

bus_eval  = load_dataset("SahmBenchmark/arabic-business-mcq_eval")["test"]
bus_train = load_dataset("SahmBenchmark/arabic-business-mcq_training_standardized")["train"]

arabic_business_dataset = concatenate_datasets([
    bus_eval.map( lambda r: _process_arabic(r, "arabic_business"), remove_columns=bus_eval.column_names),
    bus_train.map(lambda r: _process_arabic(r, "arabic_business"), remove_columns=bus_train.column_names),
])
print(f"Arabic Business loaded: {len(arabic_business_dataset)} rows (train+eval)")
print("Sample:", arabic_business_dataset[0])

Map:   0%|          | 0/167 [00:00<?, ? examples/s]

{'answer': ['d'],
 'choices': ['a', 'b', 'c', 'd', 'e'],
 'question': 'السؤال: الأخطار ........... ناتجة عن خروج المراجع عن أحكام دستور المهنة عن عمد بأن ارتكب خطأ جسيماً يجرمه القانون\n\nالخيارات:\na. المدنية\nb. التأديبية\nc. أ، ب معاً\nd. الجنائية\ne. المالية'}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 3 — Merge All Datasets into One Final Dataset
# ═══════════════════════════════════════════════════════════════════════════

# Convert the CFA/CPA Python list to a HuggingFace Dataset first
cfa_cpa_hf = Dataset.from_list(cfa_cpa_data)

# Concatenate every dataset into one unified object
final_dataset = concatenate_datasets([
    cfa_cpa_hf,
    es_multifin_dataset,
    plutus_dataset,
    bhasha_dataset,
    arabic_accounting_dataset,
    arabic_business_dataset,
])

print(f"Final dataset: {len(final_dataset)} total examples")
print("\nSource breakdown:")
from collections import Counter
for src, cnt in sorted(Counter(final_dataset["source"]).items(), key=lambda x: -x[1]):
    print(f"  {src:35s} {cnt:>5}")

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/183 [00:00<?, ? examples/s]

Map:   0%|          | 0/183 [00:00<?, ? examples/s]

{'answer': ['c'],
 'choices': ['a', 'b', 'c'],
 'question': 'السؤال: نسبة الإهلاك السنوية المتعارف عليها للآلآت والمعدات\n\nالإجابات المحتملة:\na) ٥٪\nb) ١٥ سنة\nc) ١٠٪'}

In [ ]:
# ── Dataset validation ─────────────────────────────────────────────────────
from collections import Counter

answer_counts = Counter(final_dataset["answer"])
total = len(final_dataset)

print("Answer index distribution (0=A 1=B 2=C 3=D) — should be roughly uniform:")
for idx, label in zip(range(4), ['A', 'B', 'C', 'D']):
    cnt = answer_counts.get(idx, 0)
    bar = "█" * int(cnt / total * 40)
    print(f"  {label} ({idx}): {bar} {cnt:>4}  ({cnt/total*100:.1f}%)")

# Show one sample end-to-end to verify format
print("\nSample question from final_dataset[0]:")
sample = final_dataset[0]
print(f"  source : {sample['source']}")
print(f"  question: {sample['question'][:100]}...")
for i, ch in enumerate(sample['choices']):
    marker = " ◄ correct" if i == sample['answer'] else ""
    print(f"  {'ABCD'[i]}. {ch[:60]}{marker}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-7e9b493bc4dc8b(…):   0%|          | 0.00/117k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/274 [00:00<?, ? examples/s]

Map:   0%|          | 0/274 [00:00<?, ? examples/s]

{'choices': ['a', 'b'],
 'answer': ['b'],
 'question': 'التكاليف غير المباشرة هي التكاليف التي تؤدّي بشكل مباشر إلى سلعة أو خدمة محدّدة كبدل إيجار المكتب أو راتب المحاسب أو الفوائد المترتبة عن قرض مصرفي'}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 4 — Benchmark Utility Functions
# ═══════════════════════════════════════════════════════════════════════════

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import random
import gc


def load_model_4bit(model_name: str):
    """Load a causal LM in 4-bit NF4 quantisation to fit on consumer GPUs."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    print(f"Loaded: {model_name}")
    return model, tokenizer


def free_model(model):
    """Release GPU memory after benchmarking a model."""
    del model
    gc.collect()
    torch.cuda.empty_cache()


def build_prompt(tokenizer, question: str, choices: list) -> str:
    """Build a chat-formatted MCQ prompt. Caps choices to 4."""
    labels = ['A', 'B', 'C', 'D']
    choices = list(choices)[:4]   # cap to 4 — some datasets have >4 options
    options_str = "\n".join(f"{labels[i]}. {choices[i]}" for i in range(len(choices)))
    user_msg = (
        "Answer the following multiple-choice question. "
        "Respond with ONLY the letter of the correct answer (A, B, C, or D).\n\n"
        f"Question: {question}\n\n{options_str}"
    )
    if getattr(tokenizer, "chat_template", None):
        messages = [{"role": "user", "content": user_msg}]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    # Generic fallback (Llama-2 instruction format)
    return f"[INST] {user_msg} [/INST]"


def predict_answer(model, tokenizer, question: str, choices: list) -> int:
    """
    Logit-based MCQ prediction (Legal paper method).
    Caps choices to 4, scores A/B/C/D by log-probability, returns index 0-3.
    use_cache=False avoids DynamicCache.from_legacy_cache error in transformers>=4.45.
    """
    choices = list(choices)[:4]   # cap to 4 — must match build_prompt
    option_labels = ["A", "B", "C", "D"][:len(choices)]
    option_token_ids = {
        i: tokenizer.encode(opt, add_special_tokens=False)[0]
        for i, opt in enumerate(option_labels)
    }
    prompt = build_prompt(tokenizer, question, choices)
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=2048
    ).to(model.device)

    with torch.no_grad():
        # use_cache=False: skip KV-cache to avoid DynamicCache.from_legacy_cache error
        logits = model(**inputs, use_cache=False).logits[0, -1, :]

    scores = {i: logits[tok_id].item() for i, tok_id in option_token_ids.items()}
    return max(scores, key=scores.get)


def run_benchmark(model, tokenizer, dataset, model_name: str):
    """
    Run zero-shot MCQ benchmark over the full dataset.
    Returns (accuracy_float, predictions_list_of_ints).
    Accuracy = proportion of correctly identified options.
    """
    correct = 0
    predictions = []
    n = len(dataset)

    print(f"\n{'='*60}")
    print(f"Benchmarking: {model_name}")
    print(f"Questions   : {n}")
    print(f"{'='*60}")

    for i in range(n):
        row = dataset[i]
        pred = predict_answer(model, tokenizer, row["question"], row["choices"])
        predictions.append(pred)
        if pred == row["answer"]:
            correct += 1
        if (i + 1) % 200 == 0 or (i + 1) == n:
            print(f"  [{i+1:>5}/{n}]  running accuracy: {correct/(i+1)*100:.2f}%")

    accuracy = correct / n * 100
    print(f"\n► {model_name}  —  Final Accuracy: {accuracy:.2f}%  ({correct}/{n})")
    return accuracy, predictions


# Container for all results
benchmark_results = {}

print("Benchmark utilities ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 5 — Model Registry + HuggingFace Login
#
# Llama models require accepting the licence on huggingface.co and logging in:
#   $ huggingface-cli login
#   or in-notebook:
#   from huggingface_hub import login; login(token="hf_YOUR_TOKEN")
# ═══════════════════════════════════════════════════════════════════════════

MODEL_PHI    = "microsoft/Phi-3.5-mini-instruct"
MODEL_LLAMA3 = "meta-llama/Meta-Llama-3-8B-Instruct"
MODEL_LLAMA2 = "meta-llama/Llama-2-13b-chat-hf"

print("Models to benchmark:")
for m in [MODEL_PHI, MODEL_LLAMA3, MODEL_LLAMA2]:
    print(f"  {m}")
print(f"\nTotal questions in final_dataset: {len(final_dataset)}")

## Benchmark Execution

Models are loaded **one at a time** in 4-bit NF4 quantisation.  
Each model is freed from GPU memory before the next is loaded to avoid OOM errors.

**Evaluation method**: For each question the model scores options A/B/C/D by the  
log-probability assigned to that letter token at the last input position.  
The highest-scoring letter is taken as the prediction.

### Model 1: Phi-3.5-mini-instruct (`microsoft/Phi-3.5-mini-instruct`)  
~3.8B parameters — runs on as little as 8 GB VRAM with 4-bit quantisation.

In [ ]:
model1, tokenizer1 = load_model_4bit(MODEL_PHI)

acc1, preds1 = run_benchmark(model1, tokenizer1, final_dataset, MODEL_PHI)
benchmark_results[MODEL_PHI] = {"accuracy": acc1, "predictions": preds1}

free_model(model1)
del tokenizer1

### Model 2: Llama-3-8B-Instruct (`meta-llama/Meta-Llama-3-8B-Instruct`)
~8B parameters — requires ~16 GB VRAM with 4-bit quantisation.  
Requires a HuggingFace access token (licence accepted on huggingface.co).

In [ ]:
model2, tokenizer2 = load_model_4bit(MODEL_LLAMA3)

acc2, preds2 = run_benchmark(model2, tokenizer2, final_dataset, MODEL_LLAMA3)
benchmark_results[MODEL_LLAMA3] = {"accuracy": acc2, "predictions": preds2}

free_model(model2)
del tokenizer2

### Model 3: Llama-2-13B-Chat (`meta-llama/Llama-2-13b-chat-hf`)
~13B parameters — requires ~24 GB VRAM with 4-bit quantisation.  
Requires a HuggingFace access token (licence accepted on huggingface.co).

In [ ]:
model3, tokenizer3 = load_model_4bit(MODEL_LLAMA2)

acc3, preds3 = run_benchmark(model3, tokenizer3, final_dataset, MODEL_LLAMA2)
benchmark_results[MODEL_LLAMA2] = {"accuracy": acc3, "predictions": preds3}

free_model(model3)
del tokenizer3

## Results: Accuracy Comparison

**Accuracy** = proportion of correctly identified answer labels (0–3) across all questions.  
Random baseline = 25.0 %

In [ ]:
import pandas as pd
from collections import Counter

# ── Overall accuracy table ─────────────────────────────────────────────────
rows = []
for model_name, res in benchmark_results.items():
    correct = sum(
        pred == final_dataset[i]["answer"]
        for i, pred in enumerate(res["predictions"])
    )
    rows.append({
        "Model":         model_name.split("/")[-1],
        "Full Name":     model_name,
        "Accuracy (%)":  round(res["accuracy"], 2),
        "Correct":       correct,
        "Total":         len(final_dataset),
    })

results_df = (
    pd.DataFrame(rows)
    .sort_values("Accuracy (%)", ascending=False)
    .reset_index(drop=True)
)
print("=" * 65)
print("               BENCHMARK RESULTS SUMMARY")
print("=" * 65)
print(results_df[["Model", "Accuracy (%)", "Correct", "Total"]].to_string(index=False))
print(f"\nRandom baseline (4-way MCQ): 25.00 %")

# ── Per-source breakdown ───────────────────────────────────────────────────
if benchmark_results:
    sources = sorted(set(final_dataset["source"]))
    src_indices = {
        src: [i for i, x in enumerate(final_dataset["source"]) if x == src]
        for src in sources
    }
    model_short = [m.split("/")[-1][:18] for m in benchmark_results]
    print("\n\nPer-source accuracy breakdown:")
    header = f"  {'Source':<33}" + "".join(f"{m:>20}" for m in model_short)
    print(header)
    print("  " + "-" * (len(header) - 2))
    for src in sources:
        idxs = src_indices[src]
        row_str = f"  {src:<33}"
        for model_name, res in benchmark_results.items():
            preds = res["predictions"]
            c = sum(preds[i] == final_dataset[i]["answer"] for i in idxs)
            row_str += f"{c/len(idxs)*100:>19.1f}%"
        print(row_str)

## Save Results

In [ ]:
import json

# Save full predictions + accuracies to JSON
output = {}
for model_name, res in benchmark_results.items():
    correct = sum(
        pred == final_dataset[i]["answer"]
        for i, pred in enumerate(res["predictions"])
    )
    output[model_name] = {
        "accuracy":    res["accuracy"],
        "correct":     int(correct),
        "total":       len(final_dataset),
        "predictions": res["predictions"],   # list of ints (0–3)
    }

with open("benchmark_results.json", "w") as f:
    json.dump(output, f, indent=2)
print("Saved: benchmark_results.json")

# Save summary table to CSV
if "results_df" in dir():
    results_df.to_csv("benchmark_summary.csv", index=False)
    print("Saved: benchmark_summary.csv")

print("\nAnswer key: 0=A  1=B  2=C  3=D")
print(f"Random baseline: 25.00 %")

<!-- (unused cell) -->

<!-- (unused cell) -->